# 00_open_and_inspect_svg

First step in a workflow for translating SVG text with an LLM:

- open SVG files from `svg_source_files`
- extract and inspect SVG `<text>` elements and their nested `<tspan>` fragments
- preserve meaningful whitespace from the SVG, including space-only tspans
- recombine fragmented tspans into text-level translation units
- exclude items not intended for translation, such as numeric-only labels and timeline/date markers
- save translation units as JSON in `json_files` for the next notebook, `01_translate_svg`

In [1]:
# Set directories

from pathlib import Path

PROJECT_ROOT = Path.cwd()
SVG_SOURCE_DIR = PROJECT_ROOT / "svg_source_files"
SVG_OUTPUT_DIR = PROJECT_ROOT / "svg_output_files"
JSON_DIR = PROJECT_ROOT / "json_files"

# Source/input folder should already exist
assert SVG_SOURCE_DIR.exists(), f"Missing folder: {SVG_SOURCE_DIR}"

# Output folders can be created automatically
SVG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.name)
print("SVG_SOURCE_DIR:", SVG_SOURCE_DIR.relative_to(PROJECT_ROOT.parent))
print("JSON_DIR:", JSON_DIR.relative_to(PROJECT_ROOT.parent))
print("SVG_OUTPUT_DIR:", SVG_OUTPUT_DIR.relative_to(PROJECT_ROOT.parent))

PROJECT_ROOT: tbe
SVG_SOURCE_DIR: tbe\svg_source_files
JSON_DIR: tbe\json_files
SVG_OUTPUT_DIR: tbe\svg_output_files


In [2]:
# Display svg files in source directory

svg_paths = sorted(SVG_SOURCE_DIR.glob("*.svg"))

print("Found SVG files:", len(svg_paths))
for p in svg_paths:
    print(" -", p.name)


Found SVG files: 4
 - tbe_01.svg
 - tbe_02.svg
 - tbe_03.svg
 - tbe_04.svg


## Read and parse SVG files
**NOTE:** This workflow will process *all* svg files contained in the svg_source_files directory 

In [3]:
from lxml import etree
import pandas as pd
import json
import re

SVG_NS = "http://www.w3.org/2000/svg"
NS = {"svg": SVG_NS}

def localname(tag: str) -> str:
    return tag.split("}", 1)[1] if tag and tag.startswith("{") else (tag or "")

def norm_ws(s: str) -> str:
    if s is None:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def parse_style(style: str):
    out = {}
    if not style:
        return out
    for part in style.split(";"):
        part = part.strip()
        if not part or ":" not in part:
            continue
        k, v = part.split(":", 1)
        out[k.strip()] = v.strip()
    return out

def group_label(g) -> str:
    return g.get("data-name") or g.get("id") or "g"

def compute_group_stack(el) -> list[str]:
    stack = []
    cur = el.getparent()
    while cur is not None:
        if localname(cur.tag) == "g":
            stack.append(group_label(cur))
        cur = cur.getparent()
    return list(reversed(stack))

def element_path(el) -> str:
    parts = []
    cur = el
    while cur is not None:
        tag = localname(cur.tag)
        if tag in ("svg", "g", "text", "tspan"):
            _id = cur.get("id")
            if _id:
                parts.append(f"{tag}#{_id}")
            else:
                parent = cur.getparent()
                if parent is None:
                    parts.append(tag)
                else:
                    same = [
                        c for c in parent
                        if hasattr(c, "tag") and localname(c.tag) == tag
                    ]
                    idx = same.index(cur) + 1  # 1-based index
                    parts.append(f"{tag}[{idx}]")
        cur = cur.getparent()
    return "/".join(reversed(parts))

def get_text_and_tspans(text_el):
    tspans = []
    chunks = []
    if text_el.text not in (None, ""):
        chunks.append(text_el.text)

    for t in text_el.findall(".//svg:tspan", namespaces=NS):
        t_text = t.text or ""
        if t_text != "":
            chunks.append(t_text)
        tspans.append({
            "tspan_id": t.get("id"),
            "tspan_text": t_text,
            "tspan_text_norm": norm_ws(t_text),
            "x": t.get("x"),
            "y": t.get("y"),
            "dx": t.get("dx"),
            "dy": t.get("dy"),
            "style": t.get("style"),
            "class": t.get("class"),
        })
        if t.tail not in (None, ""):
            chunks.append(t.tail)

    return "".join(chunks), tspans

def extract_text_inventory(svg_path: Path) -> pd.DataFrame:
    parser = etree.XMLParser(remove_blank_text=False, recover=True, huge_tree=True)
    tree = etree.parse(str(svg_path), parser)
    root = tree.getroot()

    rows = []
    for text_el in root.xpath("//svg:text", namespaces=NS):
        full_text, tspans = get_text_and_tspans(text_el)
        full_text_norm = norm_ws(full_text)
        if not full_text_norm:
            continue

        gstack = compute_group_stack(text_el)
        style = parse_style(text_el.get("style"))

        rows.append({
            "source_file": svg_path.name,
            "text_id": text_el.get("id"),
            "group_stack": " / ".join(gstack),
            "group_depth": len(gstack),
            "text_raw": full_text,
            "text_norm": full_text_norm,
            "has_tspans": len(tspans) > 0,
            "tspan_count": len(tspans),
            "x": text_el.get("x"),
            "y": text_el.get("y"),
            "transform": text_el.get("transform"),
            "class": text_el.get("class"),
            "style": text_el.get("style"),
            "style_font_family": style.get("font-family"),
            "style_font_size": style.get("font-size"),
            "style_text_anchor": style.get("text-anchor"),
            "element_path": element_path(text_el),
            "tspans": tspans,
        })

    df = pd.DataFrame(rows).sort_values(["group_stack", "element_path"]).reset_index(drop=True)
    return df

all_dfs = []
for p in svg_paths:
    df = extract_text_inventory(p)
    all_dfs.append(df)

    out_json = JSON_DIR / f"{p.stem}.text_extract.json"
    records = df.to_dict(orient="records")
    out_json.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"{p.name}: extracted {len(df)} text elements -> {out_json.name}")

df_all = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
print("Total extracted across files:", len(df_all))

missing_space_pattern_rows = df_all[df_all["text_norm"].str.contains(r"[a-z][A-Z]", regex=True, na=False)]
print("Likely missing-space pattern rows:", len(missing_space_pattern_rows))
missing_space_pattern_rows[["text_raw", "text_norm", "group_stack", "element_path"]]


tbe_01.svg: extracted 40 text elements -> tbe_01.text_extract.json
tbe_02.svg: extracted 104 text elements -> tbe_02.text_extract.json
tbe_03.svg: extracted 136 text elements -> tbe_03.text_extract.json
tbe_04.svg: extracted 104 text elements -> tbe_04.text_extract.json
Total extracted across files: 384
Likely missing-space pattern rows: 0


,text_raw,text_norm,group_stack,element_path


### Human-readable summary

In [4]:
def outline(df: pd.DataFrame, max_chars: int = 90):
    for i, r in df.iterrows():
        snippet = r["text_norm"]
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars-1] + "…"
        print(f"[{i:04d}] {r['group_stack']} :: {snippet}")

for fname in df_all["source_file"].unique():
    print("\n" + "="*80)
    print(fname)
    print("="*80)
    outline(df_all[df_all["source_file"] == fname].reset_index(drop=True), max_chars=110)



tbe_01.svg
[0000] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Adam :: d. 3244
[0001] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Adam :: Adam 4174
[0002] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Enoch :: 3187
[0003] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Enoch :: Enoch 3552
[0004] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Enosh :: d. 3034
[0005] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Enosh :: Enosh 3939
[0006] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Jared :: Jared 3714
[0007] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Kenan :: d. 2939
[0008] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Kenan :: Kenan 3849
[0009] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Lamech :: Lamech 3300
[0010] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patriarchs / Mahalalel :: d. 2884
[0011] AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_patr

### Create LLM-ready table of elements

In [5]:
import hashlib
import pandas as pd

def make_unit_key(source_file: str, element_path: str, text_id: str | None, source_text: str) -> str:
    # Hash stable text-level information.
    parts = [source_file, element_path]
    if text_id is not None and not pd.isna(text_id) and str(text_id).strip():
        parts.append(f"text_id={text_id}")
    parts.append(f"source_text={source_text}")
    raw = "|".join(parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def nonempty_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()

def build_translation_df(df_text: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, r in df_text.iterrows():
        source_file = r["source_file"]
        element_path = r["element_path"]
        group_stack = r["group_stack"]
        text_id = r.get("text_id")

        src = nonempty_text(r.get("text_norm"))
        if not src:
            tspans = r.get("tspans")
            if not isinstance(tspans, list):
                tspans = []
            src = "".join(nonempty_text(t.get("tspan_text_norm")) for t in tspans)
            src = " ".join(src.split())

        if not src:
            continue

        unit_key = make_unit_key(source_file, element_path, text_id, src)
        out.append({
            "unit_key": unit_key,
            "unit_type": "text",
            "source_file": source_file,
            "group_stack": group_stack,
            "element_path": element_path,
            "text_id": text_id,
            "tspan_id": None,
            "tspan_idx": None,
            "source_text": src,
        })

    columns = ["unit_key", "unit_type", "source_file", "group_stack", "element_path", "text_id", "tspan_id", "tspan_idx", "source_text"]
    df_units = pd.DataFrame(out, columns=columns)
    # helpful sorting for review
    return df_units.sort_values(["source_file", "group_stack", "element_path", "unit_type", "tspan_idx"]).reset_index(drop=True)

df_units = build_translation_df(df_all)
print("Total translation units:", len(df_units))
print("Rows containing Adam:", df_units["source_text"].str.contains("Adam", na=False).sum())
print("First 20 rows of df_units:")
df_units.head(20)


Total translation units: 384
Rows containing Adam: 1
First 20 rows of df_units:


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text
0,191f152ec4069b81f0c6059d6374f8d6bfb3dc0c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3244
1,42415ca24611ff65186249ab11a15d368b1530d6,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Adam 4174
2,d36f6b0ba1629a6c63cba162cb0023e15b3b331d,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,3187
3,63b52a9dd6f89c753d7d5daf49710893a72629c7,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enoch 3552
4,294d4a1c53b8e9c82bd7608fc412f7c1f5996ad3,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3034
5,c7cbff308d158d2ed4d94d7294e625fc2a2319dc,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enosh 3939
6,872ee56f45524087e305b6a4cdcd8d5a1fce1159,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Jared 3714
7,aaec140285126bea8a3d234be0142a830eb466b4,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 2939
8,e40bcebd3c831a8e80a629f41bf3e7104d2b251c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Kenan 3849
9,3c23d44eb081747c7ba965be21bee62ddd2676b5,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Lamech 3300


### Skip numbers and dates
Comment out if you wish to have the LLM translate the numbers or dates

In [6]:
import re

def is_numericish(s: str) -> bool:
    s = s.strip().lower()
    # numbers, punctuation, simple bc/ad, ca
    return bool(re.fullmatch(r"[0-9\s\.,:\-–—/()]*", s)) or bool(re.fullmatch(r"(ca\s*)?[0-9\s\.,\-–—/]+(bc|bce|ad|ce)?", s))

def is_short_label(s: str) -> bool:
    return len(s.strip()) <= 2

def is_death_date_marker(s: str) -> bool:
    return bool(re.fullmatch(r"d\.\s*\d+", str(s).strip(), flags=re.IGNORECASE))

df_units["skip_reason"] = ""
df_units.loc[df_units["source_text"].map(is_short_label), "skip_reason"] = "too_short"
df_units.loc[df_units["source_text"].map(is_numericish), "skip_reason"] = "numericish"
df_units.loc[df_units["source_text"].map(is_death_date_marker), "skip_reason"] = "death_date_marker"

df_units_keep = df_units[df_units["skip_reason"] == ""].copy()
df_units_skip = df_units[df_units["skip_reason"] != ""].copy()

print("Total units:", len(df_units))
print("Translate candidates:", len(df_units_keep))
print("Skipped units:", len(df_units_skip))
print("death_date_marker rows:", (df_units["skip_reason"] == "death_date_marker").sum())

df_units.loc[
    df_units["skip_reason"] == "death_date_marker",
    ["source_text", "skip_reason", "group_stack", "element_path"],
]


Total units: 384
Translate candidates: 341
Skipped units: 43
death_date_marker rows: 18


,source_text,skip_reason,group_stack,element_path
0,d. 3244,death_date_marker,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...
4,d. 3034,death_date_marker,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...
7,d. 2939,death_date_marker,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...
10,d. 2884,death_date_marker,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...
14,d. 3132,death_date_marker,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...
41,d. 1991,death_date_marker,AB2 / ab2_x5F_biblical / ab2_x5F_biblical_x5F_...,svg/g#AB2/g#ab2_x5F_biblical/g#ab2_x5F_biblica...
43,d. 2078,death_date_marker,AB2 / ab2_x5F_biblical / ab2_x5F_biblical_x5F_...,svg/g#AB2/g#ab2_x5F_biblical/g#ab2_x5F_biblica...
45,d. 1987,death_date_marker,AB2 / ab2_x5F_biblical / ab2_x5F_biblical_x5F_...,svg/g#AB2/g#ab2_x5F_biblical/g#ab2_x5F_biblica...
48,d. 1886,death_date_marker,AB2 / ab2_x5F_biblical / ab2_x5F_biblical_x5F_...,svg/g#AB2/g#ab2_x5F_biblical/g#ab2_x5F_biblica...
50,d. 1859,death_date_marker,AB2 / ab2_x5F_biblical / ab2_x5F_biblical_x5F_...,svg/g#AB2/g#ab2_x5F_biblical/g#ab2_x5F_biblica...


In [7]:
print("Duplicate unit_keys:", df_units_keep["unit_key"].duplicated().sum())

Duplicate unit_keys: 0


### Parse to JSON and save to JSON files directory
**CAUTION:** this will overwrite any previously run/saved versions

In [8]:
import json
from pathlib import Path

out_units = JSON_DIR / "translation_units.json"

if out_units.exists():
    print("Overwriting existing file:", out_units.relative_to(PROJECT_ROOT.parent))
else:
    print("Creating new file:", out_units.relative_to(PROJECT_ROOT.parent))

chars_written = out_units.write_text(
    json.dumps(df_units_keep.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Characters written:", chars_written)

Overwriting existing file: tbe\json_files\translation_units.json
Characters written: 167098


In [9]:
df_all.head(2)

,source_file,text_id,group_stack,group_depth,text_raw,text_norm,has_tspans,tspan_count,x,y,transform,class,style,style_font_family,style_font_size,style_text_anchor,element_path,tspans
0,tbe_01.svg,None,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,4,d. 3244,d. 3244,False,0,None,None,matrix(1 0 0 1 690.3776 182.9529),None,None,None,None,None,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,[]
1,tbe_01.svg,None,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,4,Adam 4174,Adam 4174,False,0,None,None,matrix(1 0 0 1 21.8434 191.8456),None,None,None,None,None,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,[]


In [10]:
df_units.head(4)

,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
0,191f152ec4069b81f0c6059d6374f8d6bfb3dc0c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3244,death_date_marker
1,42415ca24611ff65186249ab11a15d368b1530d6,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Adam 4174,
2,d36f6b0ba1629a6c63cba162cb0023e15b3b331d,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,3187,numericish
3,63b52a9dd6f89c753d7d5daf49710893a72629c7,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enoch 3552,


In [11]:
cols = [
    "unit_type",
    "source_file",
    "group_stack",
    "element_path",
    "text_id",
    "tspan_id",
    "tspan_idx",
    "source_text",
    "skip_reason",
]

df_units.loc[0:3, cols]

,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
0,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3244,death_date_marker
1,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Adam 4174,
2,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,3187,numericish
3,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enoch 3552,


In [12]:
df_units.loc[0:3, ["element_path", "tspan_idx", "source_text"]].to_string(index=False)

'                                                                  element_path tspan_idx source_text\n svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblical_x5F_patriarchs/g#Adam/text[1]      None     d. 3244\n svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblical_x5F_patriarchs/g#Adam/text[2]      None   Adam 4174\nsvg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblical_x5F_patriarchs/g#Enoch/text[1]      None        3187\nsvg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblical_x5F_patriarchs/g#Enoch/text[2]      None  Enoch 3552'

In [13]:
(
    df_units
    .sort_values(["source_file", "element_path", "tspan_idx"])
    .groupby(["source_file", "element_path"], dropna=False)["source_text"]
    .apply(lambda parts: "".join(str(x) for x in parts if str(x) != "nan"))
    .reset_index(name="combined_text")
    .head(20)
)

,source_file,element_path,combined_text
0,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,d. 3244
1,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Adam 4174
2,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,3187
3,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Enoch 3552
4,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,d. 3034
5,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Enosh 3939
6,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Jared 3714
7,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,d. 2939
8,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Kenan 3849
9,tbe_01.svg,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,Lamech 3300


In [14]:
print("Total units:", len(df_units))
print("Translate candidates:", (df_units["skip_reason"] == "").sum())
print("Skipped units:", (df_units["skip_reason"] != "").sum())

df_units.head(20)

Total units: 384
Translate candidates: 341
Skipped units: 43


,unit_key,unit_type,source_file,group_stack,element_path,text_id,tspan_id,tspan_idx,source_text,skip_reason
0,191f152ec4069b81f0c6059d6374f8d6bfb3dc0c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3244,death_date_marker
1,42415ca24611ff65186249ab11a15d368b1530d6,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Adam 4174,
2,d36f6b0ba1629a6c63cba162cb0023e15b3b331d,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,3187,numericish
3,63b52a9dd6f89c753d7d5daf49710893a72629c7,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enoch 3552,
4,294d4a1c53b8e9c82bd7608fc412f7c1f5996ad3,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 3034,death_date_marker
5,c7cbff308d158d2ed4d94d7294e625fc2a2319dc,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Enosh 3939,
6,872ee56f45524087e305b6a4cdcd8d5a1fce1159,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Jared 3714,
7,aaec140285126bea8a3d234be0142a830eb466b4,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,d. 2939,death_date_marker
8,e40bcebd3c831a8e80a629f41bf3e7104d2b251c,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Kenan 3849,
9,3c23d44eb081747c7ba965be21bee62ddd2676b5,text,tbe_01.svg,AB1 / ab1_x5F_biblical / ab1_x5F_biblical_x5F_...,svg/g#AB1/g#ab1_x5F_biblical/g#ab1_x5F_biblica...,None,None,None,Lamech 3300,
